# # SageMaker Pipeline for GenAI Model Training and Deployment

# This notebook demonstrates a complete MLOps workflow using SageMaker Pipelines. We will:
# 1.  **Prepare Data**: Create a simple dataset for fine-tuning a generative AI model.
# 2.  **Define Pipeline Steps**:
#     * **Training**: Fine-tune the `distilgpt2` model from Hugging Face.
#     * **Evaluation**: Evaluate the model's performance (using perplexity).
#     * **Conditional Registration**: Register the model in the SageMaker Model Registry only if its performance meets a specified threshold.
# 3.  **Execute the Pipeline**: Assemble the steps and run the pipeline.
# 4.  **Deploy for Inference**: Deploy the approved model from the registry to a real-time SageMaker endpoint.

## 1. Setup and Configuration

In [2]:
!pip install "sagemaker>=2.200.0" "torch" "transformers" "datasets" --upgrade --quiet

In [5]:
import sagemaker
import boto3
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.properties import PropertyFile
from sagemaker.huggingface import HuggingFace
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.parameters import (
    ParameterString,
    ParameterInteger,
)
import json
import time
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

In [6]:
sess = sagemaker.Session()
bucket = sess.default_bucket()
sagemaker_role = sagemaker.get_execution_role()

region = sess.boto_region_name
model_package_group_name = "GenAIPipelineRegistry" # A name for the group of models this pipeline will create.

print(f"SageMaker Role ARN: {sagemaker_role}")
print(f"S3 Bucket: {bucket}")
print(f"AWS Region: {region}")

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


SageMaker Role ARN: arn:aws:iam::452706865406:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole
S3 Bucket: sagemaker-us-east-1-452706865406
AWS Region: us-east-1


## 2. Prepare Data and Scripts

In [7]:
import pandas as pd

# Create a simple dataset for fine-tuning
data = {
    'text': [
        'prompt: What is the capital of France? completion: The capital of France is Paris.',
        'prompt: Who wrote "Hamlet"? completion: "Hamlet" was written by William Shakespeare.',
        'prompt: What is the formula for water? completion: The chemical formula for water is H2O.'
    ]
}
df = pd.DataFrame(data)

# Save as train and test CSV files
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

train_df.to_csv('train.csv', index=False, header=False)
test_df.to_csv('test.csv', index=False, header=False)

# ### Upload Data to S3
# The SageMaker training job will read its data from S3.

s3_input_train = sess.upload_data(path='train.csv', bucket=bucket, key_prefix='genai-pipeline/data/train')
s3_input_test = sess.upload_data(path='test.csv', bucket=bucket, key_prefix='genai-pipeline/data/test')

print(f"Training data uploaded to: {s3_input_train}")
print(f"Test data uploaded to: {s3_input_test}")

Training data uploaded to: s3://sagemaker-us-east-1-452706865406/genai-pipeline/data/train/train.csv
Test data uploaded to: s3://sagemaker-us-east-1-452706865406/genai-pipeline/data/test/test.csv


In [8]:
%%writefile requirements.txt
transformers==4.30.0
datasets==2.12.0
torch==2.0.1
evaluate
accelerate

Writing requirements.txt


In [ ]:
%%writefile train.py

import argparse
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

def train(args):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    model = AutoModelForCausalLM.from_pretrained(args.model_name)

    # Set padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load dataset
    dataset = load_dataset('csv', data_files={'train': os.path.join(args.train_data_dir, 'train.csv')})

    # Tokenize the data
    def tokenize_function(examples):
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

    tokenized_datasets = dataset.map(tokenize_function, batched=True)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir=args.model_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        save_total_limit=2,
        save_steps=500,
        logging_steps=100,
    )

    # Create Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        tokenizer=tokenizer,
    )

    # Start training
    trainer.train()

    # Save the model
    trainer.save_model(args.model_dir)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--model-name", type=str, default="distilgpt2")
    parser.add_argument("--model-dir", type=str, default=os.environ["SM_MODEL_DIR"])
    parser.add_argument("--train-data-dir", type=str, default=os.environ["SM_CHANNEL_TRAIN"])

    args = parser.parse_args()
    train(args)

Overwriting train.py


In [11]:
%%writefile evaluate.py

import argparse
import os
import json
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

def evaluate_model(args):
    # Load tokenizer and model from the training output
    tokenizer = AutoTokenizer.from_pretrained(args.model_dir)
    model = AutoModelForCausalLM.from_pretrained(args.model_dir)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load and tokenize test data
    test_dataset = load_dataset('csv', data_files={'test': os.path.join(args.test_data_dir, 'test.csv')})['test']

    def tokenize_function(examples):
        return tokenizer(examples['text'], return_tensors="pt", padding=True, truncation=True)

    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

    # Calculate perplexity
    perplexity = evaluate.load("perplexity", module_type="metric")
    results = perplexity.compute(model_id=args.model_dir, predictions=tokenized_test_dataset['text'])

    perplexity_score = results['mean_perplexity']
    print(f"Calculated Perplexity: {perplexity_score}")

    # Save evaluation report
    report_dict = {
        "metrics": {
            "perplexity": {
                "value": perplexity_score,
            },
        },
    }

    output_path = os.path.join(args.evaluation_output_dir, "evaluation.json")
    with open(output_path, "w") as f:
        json.dump(report_dict, f)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", type=str, default="/opt/ml/processing/model")
    parser.add_argument("--test-data-dir", type=str, default="/opt/ml/processing/test")
    parser.add_argument("--evaluation-output-dir", type=str, default="/opt/ml/processing/evaluation")

    args = parser.parse_args()
    evaluate_model(args)

Writing evaluate.py


## 3. Define the SageMaker Pipeline Steps

In [ ]:

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
training_instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.g5.xlarge")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")


### Step 1: Training Step

In [13]:
hyperparameters = {
    'epochs': 3,
    'batch-size': 2,
    'model-name': 'distilgpt2'
}

huggingface_estimator = HuggingFace(
    entry_point='train.py',
    source_dir='./',
    instance_type=training_instance_type,
    instance_count=1,
    role=sagemaker_role,
    transformers_version='4.28',
    pytorch_version='2.0',
    py_version='py310',
    hyperparameters=hyperparameters,
    base_job_name='genai-finetune'
)

step_train = TrainingStep(
    name="FineTuneGenAIModel",
    estimator=huggingface_estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(
            s3_data=s3_input_train, content_type="text/csv"
        )
    }
)

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


### Step 2: Evaluation Step

In [14]:
# This step runs the `evaluate.py` script to assess the model's performance.

script_evaluator = ScriptProcessor(
    image_uri=huggingface_estimator.training_image_uri(), # Use the same image as training
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="script-genai-eval",
    role=sagemaker_role,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

step_eval = ProcessingStep(
    name="EvaluateGenAIModel",
    processor=script_evaluator,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=s3_input_test,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="evaluate.py",
    property_files=[evaluation_report],
)


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


### Step 3: Registration Step

In [ ]:
from sagemaker.workflow.functions import Join
from sagemaker.model_metrics import ModelMetrics, MetricsSource

evaluation_s3_uri = Join(
    on="/",
    values=[
        step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
        "evaluation.json",
    ],
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=evaluation_s3_uri,
        content_type="application/json",
    )
)

step_register_model = RegisterModel(
    name="RegisterGenAIModel",
    model=model,
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large", "ml.g5.xlarge"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)


### Step 4: Conditional Step

In [27]:
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep

cond_lte = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,     # the name of your processing step
        property_file=evaluation_report,  # the PropertyFile object
        json_path="metrics.perplexity.value",  # path in your JSON
    ),
    right=perplexity_threshold,
)

step_conditional = ConditionStep(
    name="CheckPerplexityCondition",
    conditions=[cond_lte],
    if_steps=[step_register_model],
    else_steps=[],
)


## 4. Assemble and Execute the Pipeline

In [28]:
# Now, we define the `Pipeline` object with our steps and their dependencies, then start its execution.

pipeline_name = f"GenAI-FineTune-Deploy-Pipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        training_instance_type,
        model_approval_status,
    ],
    steps=[step_train, step_eval, step_conditional],
    sagemaker_session=sess,
)

# ### Create/Update and Run the Pipeline
print("Creating/updating and starting the pipeline...")
pipeline.upsert(role_arn=sagemaker_role)
execution = pipeline.start()

print(f"Pipeline started. Execution ARN: {execution.arn}")

# You can describe the execution to see its status
execution.describe()

# Wait for the execution to complete
print("Waiting for pipeline execution to complete...")
execution.wait()
print("Pipeline execution finished.")

Creating/updating and starting the pipeline...


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


Pipeline started. Execution ARN: arn:aws:sagemaker:us-east-1:452706865406:pipeline/GenAI-FineTune-Deploy-Pipeline/execution/bt52hq2vogyu
Waiting for pipeline execution to complete...


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:28                                                                                   │
│                                                                                                  │
│   25                                                                                             │
│   26 # Wait for the execution to complete                                                        │
│   27 print("Waiting for pipeline execution to complete...")                                      │
│ ❱ 28 execution.wait()                                                                            │
│   29 print("Pipeline execution finished.")                                                       │
│   30                                                                                             │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/workflow/pipeline.p │
│ y:985 in wait                                                                                    │
│                                                                                                  │
│    982 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    983 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    984 │   │   )                                                                                 │
│ ❱  985 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    986 │                                                                                         │
│    987 │   def result(self, step_name: str):                                                     │
│    988 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/waiter.py:58 in wait │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/context.py:123 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                               

## 5. Deploy the Registered Model to an Endpoint

In [ ]:

# After the pipeline successfully completes and the model is registered (and manually approved, if you set `PendingManualApproval`), you can deploy it from the Model Registry.

# ### Find the Latest Approved Model Package

sm_client = boto3.client("sagemaker", region_name=region)

# Find the latest approved model package from our model group
response = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    ModelApprovalStatus='Approved', # Change to 'PendingManualApproval' if you haven't approved it yet
    SortBy='CreationTime',
    SortOrder='Descending'
)

if not response['ModelPackageSummaryList']:
    print(f"No approved models found in group: {model_package_group_name}. Please approve a model in the SageMaker console.")
else:
    latest_model_package_arn = response['ModelPackageSummaryList'][0]['ModelPackageArn']
    print(f"Latest approved model package: {latest_model_package_arn}")

    # ### Create a SageMaker Model from the Model Package
    model_name = f"genai-model-{int(time.time())}"

    # Use the HuggingFaceModel class for easy deployment
    from sagemaker.huggingface.model import HuggingFaceModel

    model_for_deployment = HuggingFaceModel(
       model_data=latest_model_package_arn, # The key difference: use the ARN from registry
       role=sagemaker_role,
       transformers_version="4.28",
       pytorch_version="2.0",
       py_version="py310"
    )

    # ### Deploy to a Real-time Endpoint
    endpoint_name = f"genai-endpoint-{int(time.time())}"

    print(f"Deploying model to endpoint: {endpoint_name}")

    predictor = model_for_deployment.deploy(
        initial_instance_count=1,
        instance_type="ml.g5.xlarge", # Use a GPU instance for good performance
        endpoint_name=endpoint_name
    )

    print(f"Endpoint {endpoint_name} is now in service.")

    # ### Invoke the Endpoint for Inference 🧠
    prompt = "prompt: What is the capital of France? completion:"
    data = {"inputs": prompt}

    response = predictor.predict(data)
    generated_text = response[0]['generated_text']

    print("\n--- Inference Result ---")
    print(f"Prompt: {prompt}")
    print(f"Generated Text: {generated_text}")
    print("----------------------")

In [ ]:
### Clean Up
# Remember to delete the endpoint to avoid incurring costs when you are done.
predictor.delete_endpoint()
print(f"\nEndpoint {endpoint_name} deleted.")